# Mean-Field Simulation of a Spin–Orbit Coupled Quantum Dot

This notebook implements a mean-field calculation of Andreev bound states (ABSs)
and the Josephson current in a superconducting quantum dot with Zeeman field, spin–orbit coupling and Coulomb interactions. In this case l=2, corresponding to positive-energy 4 spin-split ABSs. Here no bosonic bath is present.

The workflow is:

1. Define model parameters
2. Construct dot eigenstates
3. Compute coupling matrices
4. Calculate Andreev bound states
5. Solve the mean-field equations
6. Compute the Josephson current

Note: this notebook illustrates the implementation of a mean-field simulation. Results have not been benchmarked against reference data and are provided for demonstration purposes..

### Imports

In [ ]:
import math
import cmath

import numpy as np
from numpy import linalg as LA

import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy import integrate
from scipy.linalg import null_space
from scipy.optimize import fsolve
from scipy.integrate import solve_bvp

from joblib import Parallel, delayed
import multiprocessing

# number of parallel processes for parallel computation
n_jobs = multiprocessing.cpu_count()

In [ ]:
%matplotlib notebook

### Parameters

In [ ]:
# system length
L = 1.7

# number of orbitals
l = 2

# coupling to superconducting leads
t_L = 1
t_R = t_L

# effective mass and superconducting gap
mx = 9
Delta = 1

# continuum temperature
T_ferm = 0.3

# Matsubara frequencies
ferm_freq = 1j * np.pi * T_ferm * np.arange(-1001, 1003, 2)

# Zeeman field
b_x = 0.001
b_z = 0.0

# spin-orbit coupling
gamma_SO = 0.0001

# Coulomb interaction and gate voltage
Ec = 0.95
Vg_min_mu = -1

### Phase grid

In [ ]:
phi_0 = np.linspace(0, np.pi, num=41)

### Pauli matrices

In [ ]:
tau_z = np.block([
    [np.eye(4), np.zeros((4,4))],
    [np.zeros((4,4)), -np.eye(4)]])

### Initialization of Mean-Field Parameters

In [ ]:
MF_param = np.zeros((len(phi_0), 2*l))

accur_par = 10**(-3)
abs_diff = np.full((len(phi_0), 2*l), np.inf)

### Fermi distribution for the continuum

In [ ]:
def fermi_distrib(x, T):
    return 1/(np.exp(x/T)+1)

## Finding free dot eigenenergies and eigenfunctions -  hopping parameters

The eigenfunctions of the isolated dot are calculated including
spin–orbit coupling.<br> Lead-dot hopping parameters are constructed from them.

### Construction of the free dot Hamiltonian

In [ ]:
def chi_n_sigma(x, n, sigma):
    """
    Eigenfunction of the isolated dot including spin–orbit coupling.
    """
    
    q = mx*gamma_SO
    k_n = n*np.pi/L
    
    cos_n = (k_n + sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    sin_n = (k_n - sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    
    f = np.exp(-1j*sigma*q*(x - L/2)) * (cos_n * np.exp(1j*k_n*(x-L/2)) + \
                                   sin_n * np.exp(-1j*k_n*(x-L/2)))
    
    return f/np.sqrt(L)

In [ ]:
# Example wavefunction plot

x_plot = np.linspace(-L/2, L/2, 100)

plt.xlabel("x")
plt.ylabel("Wavefunction")


plt.plot(x_plot, np.real(chi_n_sigma(x_plot, 3, -1)))
plt.plot(x_plot, np.imag(chi_n_sigma(x_plot, 3, -1)))

plt.grid()

In [ ]:
def cos_n_s(n, sigma):
    """
    Plane-wave mixing coefficient in the SO-coupled dot eigenstate.
    """
    
    k_n = n*np.pi/L
    q = mx*gamma_SO
    
    cos_n = (k_n + sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    
    return cos_n   

In [ ]:
def J_mn_sigma(m, n, sigma):
     """
    Spin-flip coupling between orbitals m and n induced by
    spin–orbit interaction and transverse Zeeman field.
    """
    
    q = mx*gamma_SO
    k_n = n*np.pi/L
    k_m = m*np.pi/L
    
    f_s_mn = cos_n_s(n, sigma) * cos_n_s(m, -sigma) / (k_n - k_m - 2*q*sigma) -\
             cos_n_s(n, -sigma) * cos_n_s(m, sigma) / (k_n - k_m + 2*q*sigma) +\
             cos_n_s(n, sigma) * cos_n_s(m, sigma) / (k_n + k_m - 2*q*sigma) -\
             cos_n_s(n, -sigma) * cos_n_s(m, -sigma) / (k_n + k_m + 2*q*sigma)
    
    if (-1)**(n+m) == 1:
        J = 2*b_x * f_s_mn / L * (-sigma)*np.sin(q*L)
    if (-1)**(n+m) == -1:
        J = 2*b_x * f_s_mn / L * (-1j)*np.cos(q*L)
    
    return J

In [ ]:
# number of dot levels used in the truncated basis
N = 20

# orbital indices
n_arr = np.arange(1, N + 1)

# spin–orbit momentum shift
q = mx * gamma_SO

# quantized momenta of the dot
k_n = n_arr * np.pi / L

# single-particle energies of the isolated dot
eps_n = k_n**2 / (2 * mx) - mx * gamma_SO**2 / 2 + Vg_min_mu


# spin-flip coupling matrices between orbitals
J_min = np.zeros((N, N), dtype='complex')
J_pl = np.zeros_like(J_min)

# compute coupling elements J_{mn}
for n in range(N):
    for m in range(N):
        J_min[n, m] = J_mn_sigma(n+1, m+1, -1)
        J_pl[m, n] = np.conjugate(J_min[n, m])

In [ ]:
# effective Hamiltonian
H_2 = np.zeros((2*N, 2*N), dtype='complex')

diagonal_elements = np.array([(eps + b_z, eps - b_z) for eps in eps_n]).flatten()
H_diag = np.diag(diagonal_elements)

for n in range(N):
    for m in range(N):
        H_2[2*n, 2*m+1] = J_min[n, m]
        H_2[2*n+1, 2*m] = J_pl[n, m]
        
H_2 += H_diag

# finding the eigenenergies and eigenfunctions
E_2, Psi_2 = LA.eigh(H_2)

for i in range(2*N):
    sum_i = np.real(np.sum(Psi_2[:, i]**2))
    Psi_2[:, i] /= sum_i

# hopping parameters    
even_rows = Psi_2[::2, :]
u = np.sum(xi_n[:, np.newaxis] * even_rows, axis=0)
t_L_up = np.exp(1j*L*q) * np.sum(m_one_arr[:, np.newaxis] * xi_n[:, np.newaxis] * even_rows, axis=0)


odd_rows = Psi_2[1::2, :]
v = np.sum(xi_n[:, np.newaxis] * odd_rows, axis=0)
t_L_down = np.exp(-1j*L*q) * np.sum(m_one_arr[:, np.newaxis] * xi_n[:, np.newaxis] * odd_rows, axis=0)

In [ ]:
# printing the relevant eigenenergies

print(E_2[0:2*l])

In [ ]:
# left- and right-leads hybridization matrices

Gamma_L = np.zeros((4, 4), dtype='complex')
Gamma_R = np.zeros_like(Gamma_L)
F_R = np.zeros_like(Gamma_L)
F_L = np.zeros_like(Gamma_L)

for mu in range(4):
    for nu in range(4):
        Gamma_R[mu, nu] = np.conjugate(u[mu])*u[nu] + np.conjugate(v[mu])*v[nu]
        Gamma_L[mu, nu] = u[mu]*np.conjugate(u[nu]) + v[mu]*np.conjugate(v[nu])
    
        F_R[mu, nu] = u[mu]*v[nu] - u[nu]*v[mu]
        F_L[mu, nu] = np.conjugate(u[mu]*v[nu] - u[nu]*v[mu])  

Gamma_L *= t_L**2
Gamma_R *= t_L**2
F_L *= t_L**2
F_R *= t_L**2

## Mean-field Hamiltonian

In [ ]:
def det_G_inv(x, i, MF_param):
    """
    Computes det(G^{-1}(x)) for the interacting quantum dot.

    The inverse Green function includes the dot Hamiltonian,
    mean-field interaction terms, and the superconducting
    self-energy from the left and right leads.

    The zeros of this determinant correspond to the
    Andreev bound state energies.
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    
    Lambda = -1 / cmath.sqrt(Delta**2 - x**2) \
            * (x * np.block([[(Gamma_L + Gamma_R), zero_m], [zero_m, np.conjugate(Gamma_L + Gamma_R)]]) \
            + Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) \
            + Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))
    
    G_inv = x * np.eye(4*l) - np.block([[np.diag(E_2[0:2*l]) - 2*1j*Ec*np.diag(MF_param[i]), zero_m],\
                                        [zero_m, -np.diag(E_2[0:2*l]) + 2*1j*Ec*np.diag(MF_param[i])]]) - Lambda
    
    return LA.det(G_inv)   

Here algorithmically one does not need to compute the ABS energies. But one can compute them to see the structure of the ABSs.

In [ ]:
# ABSs array
roots = np.zeros((len(phi_0), 4*l))

for i in range(len(phi_0)):
    roots[i, 0] = fsolve(det_G_inv, 0.3, args=(i, MF_param))  
    roots[i, 1] = fsolve(det_G_inv, 0.8, args=(i, MF_param))  
    roots[i, 2] = fsolve(det_G_inv, 0.9908, args=(i, MF_param)) 
    roots[i, 3] = fsolve(det_G_inv, 0.9999, args=(i, MF_param))
    
    roots[i, 4] = fsolve(det_G_inv, -0.7, args=(i, MF_param))  
    roots[i, 5] = fsolve(det_G_inv, -0.8, args=(i, MF_param))     
    roots[i, 6] = fsolve(det_G_inv, -0.9, args=(i, MF_param))  
    roots[i, 7] = fsolve(det_G_inv, -0.9999, args=(i, MF_param))      

In [ ]:
# ploting ABSs

plt.plot(phi_0/np.pi, roots[:,0], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,1], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,2], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,3], linewidth=1.5)


plt.title(r"$t_L={}, t_R={}, \gamma_{{SO}} = {},$"\
          .format(t_L, t_R, gamma_SO)+'\n'+
          r"$b_x={}, b_z = {}, L={}$"\
          .format(np.round(b_x, 2), np.round(b_z, 2), L), fontsize=13)



plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$E_A$, $\Delta$', fontsize=12)

In [ ]:
def G_inv(x, i, MF_param):
    """
    Computes the inverse Green's fucntion G^{-1}(x) for the interacting quantum dot.

    The inverse Green function includes the dot Hamiltonian,
    mean-field interaction terms, and the superconducting
    self-energy from the left and right leads.
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    
    Lambda = -1 / cmath.sqrt(Delta**2 - x**2) \
            * (x * np.block([[(Gamma_L + Gamma_R), zero_m], [zero_m, np.conjugate(Gamma_L + Gamma_R)]]) \
            + Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) \
            + Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))
    
    G_inv = x * np.eye(4*l) - np.block([[np.diag(E_2[0:2*l]) - 2*1j*Ec*np.diag(MF_param[i]), zero_m],\
                                        [zero_m, -np.diag(E_2[0:2*l]) + 2*1j*Ec*np.diag(MF_param[i])]]) - Lambda
    
    return G_inv  

## Mean-field parameter calculation

In [ ]:
# Projection operators used to extract auxiliary mean-field
# expectation values from the Matsubara Green function.
# Each matrix selects a specific combination of orbital occupations.

P_mn = np.array([np.diag(np.array([1, -1, 0, 0, 1, -1, 0, 0])),\
                 np.diag(np.array([1, 0, -1, 0, 1, 0, -1, 0])),\
                 np.diag(np.array([1, 0, 0, -1, 1, 0, 0, -1])),\
                 np.diag(np.array([0, 1, -1, 0, 0, 1, -1, 0])),\
                 np.diag(np.array([0, 1, 0, -1, 0, 1, 0, -1])),\
                 np.diag(np.array([0, 0, 1, -1, 0, 0, 1, -1]))])

In [ ]:
def MS_mf_param(i, MF_param):
    """
    Computes the Matsubara sum of mean-field observables
    for phase index i.

    The trace Tr[G P_mu tau_z] is evaluated for each
    Matsubara frequency, producing six auxiliary
    expectation values used to update the MF parameters.
    """
    
    Mats_sum = np.zeros(6, dtype='complex')
    
    for j in range(len(ferm_freq)):
        GF_j = G_inv(ferm_freq[j], i, MF_param)
        inv_GF_j = LA.inv(GF_j)

        for mu in range(6):
            Mats_sum[mu] += np.trace(inv_GF_j @ P_mn[mu] @ tau_z)
    
    return Mats_sum

In [ ]:
# Helper function used for parallel evaluation of the
# Matsubara mean-field observables for a single phase index.

def compute_single_MF(i, MF_param):
    return MS_mf_param(i, MF_param)

### Mean-field algorithm diagnostic

In [ ]:
# ------------------------------------------------------------
# Diagnostic cell: single mean-field update
# Used to test convergence behavior during development.
# Not part of the final MFA loop.
# Optional: decomment and run a single MF update by hand to inspect convergence behavior
# ------------------------------------------------------------

# MF_param_matr = np.array(
#     Parallel(n_jobs=n_jobs)(
#         delayed(compute_single_MF)(i, MF_param) for i in range(len(phi_0))
#     )
# )

# MF_param_matr *= T_ferm/2

# # Transformation matrix: 4 outputs as linear combinations of 6 inputs

# T = np.array([
#     [ 1,  1,  1,  0,  0,  0],
#     [-1,  0,  0,  1,  1,  0],
#     [ 0, -1,  0, -1,  0,  1],
#     [ 0,  0, -1,  0, -1, -1]])

# MF_param_new = MF_param_matr @ T.T  # (n, 6) × (6, 4) → (n, 4)

# #  checking convergence
# abs_diff = np.abs(MF_param_new - MF_param)

# # update the MF parameter
# MF_param = MF_param_new

In [ ]:
# # plotting the MF parameters

# plt.plot(phi_0/np.pi, MF_param_new)

In [ ]:
# # plotting the absolute difference

# plt.plot(phi_0/np.pi, abs_diff)

### Mean-field approximation loop

In [ ]:
# ------------------------------------------------------------
# Mean-field self-consistency loop
# ------------------------------------------------------------

# initialize MF parameters
MF_param = np.zeros((len(phi_0), 2*l), dtype="complex")

# convergence criterion
accur_par = 1e-2
abs_diff = np.full((len(phi_0), 2*l), np.inf, dtype="float64")

MFA_count = 0
max_iter = 15

# mask indicating which phase points have not yet converged
active_mask = np.full(len(phi_0), True)

while np.any(abs_diff[active_mask] > accur_par):

    if MFA_count >= max_iter:
        print("Reached max iterations.")
        break

    print(f"\n--- Iteration {MFA_count} ---")

    # indices of phase values that still require updates
    active_indices = np.where(active_mask)[0]

    # compute MF contributions only for non-converged phase points
    MF_param_matr_partial = np.array(
        Parallel(n_jobs=n_jobs)(
            delayed(compute_single_MF)(i, MF_param) for i in active_indices
        )
    )

    MF_param_matr_partial *= T_ferm / 2

    # transformation from 6 auxiliary parameters to 4 MF parameters
    T = np.array([
        [ 1,  1,  1,  0,  0,  0],
        [-1,  0,  0,  1,  1,  0],
        [ 0, -1,  0, -1,  0,  1],
        [ 0,  0, -1,  0, -1, -1]
    ])

    MF_param_new_partial = MF_param_matr_partial @ T.T

    # check convergence
    abs_diff_partial = np.abs(
        MF_param_new_partial - MF_param[active_indices]
    )
    abs_diff[active_indices] = abs_diff_partial

    # update MF parameters
    MF_param[active_indices] = MF_param_new_partial

    # update mask of unconverged phase points
    active_mask = np.any(abs_diff > accur_par, axis=1)

    print("Remaining points to converge:", np.sum(active_mask))

    MFA_count += 1

### Plotting the MFA results

In [ ]:
# plotting the resulting MF parameters

plt.plot(phi_0/np.pi, MF_param)

plt.xlabel(r'$\varphi_0$, $\pi$', fontsize=12)
plt.ylabel(r'$X_{\mu}$, MF parameters',  fontsize=12)
plt.grid()

In [ ]:
# plotting the absilute differences

plt.plot(phi_0/np.pi, abs_diff)

plt.xlabel(r'$\varphi_0$, $\pi$', fontsize=12)
plt.ylabel(r'$Abs differences',  fontsize=12)
plt.grid()

## Calculating Josephson current and SCD efficiency for the obtained MF parameters

In [ ]:
def deriv_Lambd(x, i):
    """
    Computes the derivative of the superconducting self-energy Λ with
    respect to the superconducting phase φ₀ for phase index i.

    This quantity is used in the calculation of the Josephson current,
    since the current depends on the phase derivative of the lead
    self-energy contributions.
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    deriv_L = Delta / cmath.sqrt(Delta**2 - x**2) / (2*1j) * \
            (s_L * np.block([[zero_m, -np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))],\
                             [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) + \
             s_R * np.block([[zero_m, -np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))],\
                             [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))
    
    return deriv_L

In [ ]:
def MS_current(i, MF_param):
    """
    Computes the Matsubara sum contribution to the Josephson current
    for a given phase index i.

    The current is obtained from the trace of the Green function
    multiplied by the phase derivative of the superconducting
    self-energy Λ over all fermionic Matsubara frequencies.
    """
    
    I = 0
    for j in range(len(ferm_freq)):
        I += np.trace(LA.inv(G_inv(ferm_freq[j], i, MF_param)) @ deriv_Lambd(ferm_freq[j], i))
        
    return I

In [ ]:
# Helper function used for parallel evaluation of the
# Josephson current for a single phase index.

def compute_single_curr(i):
    return MS_current(i, MF_param)

In [ ]:
# Parallel calculation of the Josephson current

I_phi = np.array(
    Parallel(n_jobs=n_jobs)(
        delayed(compute_single_curr)(i) for i in range(len(phi_0))))

I_phi *= T_ferm

In [ ]:
# Plotting the Josephson current as a function of the superconducting phase φ₀

plt.plot(phi_0/np.pi, I_phi)
plt.grid()
plt.xlabel(r"$\varphi_0, \pi$")
plt.ylabel("I")
plt.title(r"$\Delta={},\gamma_{{SO}} = {}, b_x={}, b_z = {},$"\
          .format(Delta, gamma_SO, np.round(b_x, 2), np.round(b_z, 2))+'\n'+
          r"$ L={}, E_C = {}, V_g-\mu = {}$"\
          .format(L, Ec, Vg_min_mu), fontsize=13)

In [ ]:
# Computing the supercinducting diode efficience from the Josephson current η 

I_pl = np.max(I_phi)
I_min = np.max(-I_phi)

eta = np.abs(I_pl - I_min)/(I_pl + I_min)
print('eta = ', np.real(eta)*100, "%")